In [20]:
import pandas as pd
import numpy as np
from functions.eval import *
from scipy.stats import wilcoxon

In [ ]:
model_pred_col = "Camelbert-MSA"
model_pred_col2 = "AraBert"

In [ ]:
lime_eval = pd.read_csv("data/xai_eval/xai_eval_lime_" + model_pred_col + ".csv")
lime_eval2 = pd.read_csv("data/xai_eval/xai_eval_lime_" + model_pred_col2 + ".csv")
lime_eval = lime_eval[['LIME_combined']]
lime_eval2 = lime_eval2[['LIME_combined']]

In [ ]:
lime_eval = pd.concat([lime_eval, lime_eval2], axis=0)

In [ ]:
ensemble_eval = pd.read_csv("data/xai_eval/xai_eval_ensemble_" + model_pred_col + ".csv")
ensemble_eval2 = pd.read_csv("data/xai_eval/xai_eval_ensemble_" + model_pred_col2 + ".csv")
ensemble_eval = ensemble_eval[['EXAI_LIME_SHAP_mean_combined']]
ensemble_eval2 = ensemble_eval2[['EXAI_LIME_SHAP_mean_combined']]

In [26]:
ensemble_eval = pd.concat([ensemble_eval, ensemble_eval2], axis=0)

In [27]:
lime_eval["LIME_combined"].isna().sum(), ensemble_eval["EXAI_LIME_SHAP_mean_combined"].isna().sum()

(np.int64(2), np.int64(3))

In [28]:
# index of the rows with NaN values in the combined column
lime_eval[lime_eval["LIME_combined"].isna()].index

Index([303, 303], dtype='int64')

In [29]:
ensemble_eval[ensemble_eval["EXAI_LIME_SHAP_mean_combined"].isna()].index

Index([889, 829, 889], dtype='int64')

In [30]:
eval_df = pd.concat([lime_eval, ensemble_eval], axis=1)

In [31]:
eval_df.dropna(inplace=True)

In [32]:
eval_df.isna().sum()

LIME_combined                   0
EXAI_LIME_SHAP_mean_combined    0
dtype: int64

In [33]:
eval_df.shape

(6061, 2)

In [34]:
effect = np.mean(eval_df["EXAI_LIME_SHAP_mean_combined"] - eval_df["LIME_combined"])
effect

np.float64(0.011938422040461043)

In [35]:
stat, p = wilcoxon(eval_df["LIME_combined"].values, eval_df["EXAI_LIME_SHAP_mean_combined"].values)
print(stat, p)

6726748.0 2.3927696235425627e-66


In [36]:
def bootstrap_ci(a, b, n=10000):
    diffs = []
    for _ in range(n):
        idx = np.random.choice(len(a), len(a), replace=True)
        diffs.append(np.mean(a[idx] - b[idx]))
    return np.percentile(diffs, [2.5, 97.5])

In [37]:
bootstrap_ci(eval_df["EXAI_LIME_SHAP_mean_combined"].values, eval_df["LIME_combined"].values)

array([0.00982989, 0.01408456])